# `PCPFLAREINV` approximate inverses

This notebook explores the approximate inverses available in `PCPFLAREINV`. We will focus specifically on GMRES polynomials, as they form the foundation for the AIRG method discussed in subsequent notebooks.

GMRES polynomials can be expressed in several forms. We begin with the monomial form:
$$
p_m(A)=\sum_{k=0}^{m} c_k A^k,
$$
where the coefficients $c_k$ are chosen to minimize the 2-norm of the residual for a given right-hand side.

Standard GMRES generates a different, non-stationary polynomial at each iteration, requiring dot products for orthogonalization. In contrast, `PCPFLAREINV` constructs a fixed-order, stationary polynomial based on a single random right-hand side, which is then reused as an approximate inverse.

The primary advantage of this approach is that once the polynomial coefficients are determined, applying the approximate inverse requires only matrix-vector products.

`PCPFLAREINV` defaults used here:
- Type: GMRES polynomial (`arnoldi`)
- Polynomial order: 6
- Assembled application with fixed sparsity order 1

In [ ]:
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

import petsc4py
petsc4py.init(sys.argv)
from petsc4py import PETSc

import pflare

print('Ready.')

## 1. Build a test matrix

We use a simple, structured 2D upwind advection operator as our test problem.

In [ ]:
def build_2d_upwind(N):
    """Pure 2D upwind advection u_x + u_y = 1 on [0,1]^2."""
    h = 1.0 / (N + 1)
    n = N * N

    A = PETSc.Mat().create()
    A.setSizes([n, n])
    A.setFromOptions()
    A.setPreallocationNNZ(3)
    A.setUp()

    rstart, rend = A.getOwnershipRange()
    for row in range(rstart, rend):
        i, j = divmod(row, N)
        A.setValue(row, row, 2.0 / h)
        if j > 0:
            A.setValue(row, row - 1, -1.0 / h)
        if i > 0:
            A.setValue(row, row - N, -1.0 / h)

    A.assemblyBegin()
    A.assemblyEnd()

    b = A.createVecRight()
    b.set(1.0)
    return A, b

N = 40
A, b = build_2d_upwind(N)
n = A.getSize()[0]
nnz_A = A.getInfo()['nz_used']
print(f'A: size={n}, nnz={nnz_A:.0f}')

## 2. Utilities

When comparing assembled and matrix-free GMRES polynomials, it is important to evaluate the equivalent matrix-vector product costs. For assembled matrices, one application of the approximate inverse scales with the number of non-zeros (nnz) retained by the fixed sparsity pattern. For matrix-free applications, each polynomial order requires one matrix-vector product with the original matrix.

For consistency across all examples, all KSP solves in this notebook use **unpreconditioned residual norms** for both monitoring and stopping tests.

In [ ]:
def _set_prefixed_options(prefix, options_dict):
    opts = PETSc.Options()
    for key, value in options_dict.items():
        full_key = prefix + key
        opts[full_key] = str(value)

def _clear_prefixed_options(prefix, options_dict):
    opts = PETSc.Options()
    for key in options_dict.keys():
        full_key = prefix + key
        try:
            del opts[full_key]
        except Exception:
            pass

def run_solver_with_history(
    A, b, ksp_type='gmres', pc_type='none', pflare_opts=None,
    max_it=200, rtol=1e-10
):
    """Run KSP and return relative residual history and iteration count."""
    pflare_opts = pflare_opts or {}
    prefix = f'smooth_{int(time.time()*1e6)%10_000_000}_'

    ksp = PETSc.KSP().create()
    ksp.setOptionsPrefix(prefix)
    ksp.setOperators(A)
    ksp.setType(ksp_type)
    ksp.setTolerances(rtol=rtol, max_it=max_it)
    ksp.getPC().setType(pc_type)

    _set_prefixed_options(prefix, pflare_opts)

    rnorms = []
    def monitor(ksp_obj, it, rnorm):
        rnorms.append(rnorm)
    ksp.setMonitor(monitor)

    ksp.setFromOptions()
    ksp.setNormType(PETSc.KSP.NormType.UNPRECONDITIONED)
    x = A.createVecRight()
    ksp.solve(b, x)

    _clear_prefixed_options(prefix, pflare_opts)

    rnorms = np.array(rnorms, dtype=float)
    if rnorms.size > 0 and rnorms[0] != 0.0:
        rrel = rnorms / rnorms[0]
    else:
        rrel = rnorms
    return rrel, ksp.getIterationNumber(), ksp

class FixedGMRESApplyPC:
    """Python PC that applies a fixed number of GMRES iterations as M^{-1}."""
    def __init__(self, A, gmres_steps=6):
        self.A = A
        self.gmres_steps = int(gmres_steps)
        self.inner = PETSc.KSP().create(comm=A.getComm())
        self.inner.setOperators(A)
        self.inner.setType('gmres')
        self.inner.getPC().setType('none')
        self.inner.setGMRESRestart(self.gmres_steps)
        self.inner.setTolerances(rtol=0.0, atol=0.0, max_it=self.gmres_steps)
        self.inner.setFromOptions()
        self.inner.setNormType(PETSc.KSP.NormType.UNPRECONDITIONED)

    def apply(self, pc, x, y):
        y.set(0.0)
        self.inner.solve(x, y)

def run_richardson_with_fixed_gmres_apply(A, b, gmres_steps=6, max_it=200, rtol=1e-10):
    """Outer undamped Richardson with inner fixed-step GMRES apply as preconditioner."""
    ksp = PETSc.KSP().create(comm=A.getComm())
    ksp.setOperators(A)
    ksp.setType('richardson')
    ksp.setTolerances(rtol=rtol, max_it=max_it)

    pc = ksp.getPC()
    pc.setType(PETSc.PC.Type.PYTHON)
    pc.setPythonContext(FixedGMRESApplyPC(A, gmres_steps=gmres_steps))

    rnorms = []
    def monitor(ksp_obj, it, rnorm):
        rnorms.append(rnorm)
    ksp.setMonitor(monitor)

    ksp.setFromOptions()
    ksp.setNormType(PETSc.KSP.NormType.UNPRECONDITIONED)
    x = A.createVecRight()
    ksp.solve(b, x)

    rnorms = np.array(rnorms, dtype=float)
    if rnorms.size > 0 and rnorms[0] != 0.0:
        rrel = rnorms / rnorms[0]
    else:
        rrel = rnorms
    return rrel, ksp.getIterationNumber(), ksp

def assembled_pattern_nnz_from_sparsity_order(A, sparsity_order, cache=None):
    """Estimate assembled inverse nnz from fixed sparsity order using pattern of A^s."""
    if sparsity_order < 1:
        raise ValueError('sparsity_order must be >= 1')

    if cache is not None and sparsity_order in cache:
        return cache[sparsity_order]

    if sparsity_order == 1:
        nnz = int(A.getInfo()['nz_used'])
        if cache is not None:
            cache[sparsity_order] = nnz
        return nnz

    A_pow = A.copy()
    for _ in range(2, sparsity_order + 1):
        A_next = A_pow.matMult(A)
        A_pow.destroy()
        A_pow = A_next

    nnz = int(A_pow.getInfo()['nz_used'])
    A_pow.destroy()

    if cache is not None:
        cache[sparsity_order] = nnz
    return nnz

def richardson_matvec_equiv_per_iter(nnz_A, nnz_P=None, poly_order=None, matrix_free=False):
    if matrix_free:
        if poly_order is None:
            raise ValueError('poly_order required for matrix-free accounting')
        return 1.0 + float(poly_order)
    if nnz_P is None:
        raise ValueError('nnz_P required for assembled accounting')
    return 1.0 + float(nnz_P) / float(nnz_A)

In [ ]:
def plot_histories_vs_work(curves, title, xlabel='Matvec-equivalent work'):
    fig, ax = plt.subplots(figsize=(8, 5))
    for label, work_x, rel_res in curves:
        if len(rel_res) == 0:
            continue
        ax.semilogy(work_x, rel_res, marker='o', ms=3, label=label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Relative residual')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Assembled 6th order, 1st order fixed sparsity GMRES polynomial vs GMRES(6)

In this section, we compare two different preconditioners applied within an outer undamped Richardson iteration:
- Assembled, fixed sparsity GMRES polynomial in `PCPFLAREINV` (order 6, sparsity 1)
- GMRES(6)

While both approaches utilize 6th-order GMRES polynomials, their implementations differ significantly. By default, `PCPFLAREINV` applies an assembled matrix approximation of the 6th-order polynomial, restricted to a first-order sparsity pattern.

For example, with a 3rd-order monomial, the exact polynomial form is:
$$
p_3(A)=c_0 I + c_1 A + c_2 A^2 + c_3 A^3.
$$

With assembled fixed-sparsity construction (`-pc_pflareinv_sparsity_order=1`), higher powers are truncated back to the sparsity pattern of $A$, e.g.:
$$
\tilde p_3(A)=c_0 I + c_1 A + c_2\Pi_{\mathcal S_1}(A^2) + c_3\Pi_{\mathcal S_1}(A^3).
$$

This comparison addresses the following question: for a given number of outer Richardson iterations, how does an assembled stationary GMRES polynomial perform relative to non-stationary GMRES(6)?

In [ ]:
# Outer undamped Richardson + assembled PCPFLAREINV
r_rich_pflare, it_rich_pflare, _ = run_solver_with_history(
    A, b, ksp_type='richardson', pc_type='pflareinv', max_it=200, rtol=1e-10
 )

# Outer undamped Richardson + fixed GMRES(6) apply
r_rich_gmres6, it_rich_gmres6, _ = run_richardson_with_fixed_gmres_apply(
    A, b, gmres_steps=6, max_it=200, rtol=1e-10
 )

iters_pflare = np.arange(1, len(r_rich_pflare) + 1, dtype=float)
iters_gmres6 = np.arange(1, len(r_rich_gmres6) + 1, dtype=float)

print(f'Richardson+PCPFLAREINV iterations: {it_rich_pflare}')
print(f'Richardson+GMRES(6)-apply iterations: {it_rich_gmres6}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(iters_pflare, r_rich_pflare, marker='o', ms=3, label='Richardson + PCPFLAREINV (assembled)')
ax.semilogy(iters_gmres6, r_rich_gmres6, marker='o', ms=3, label='Richardson + GMRES(6) apply')
ax.set_xlabel('Richardson iteration')
ax.set_ylabel('Relative residual')
ax.set_title('Assembled (6,1) vs GMRES(6) - by convergence')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# Keep a plain GMRES baseline for later Section 6 work-normalized comparisons
r_gmres, it_gmres, _ = run_solver_with_history(
    A, b, ksp_type='gmres', pc_type='none', max_it=200, rtol=1e-10
 )
work_gmres = np.arange(1, len(r_gmres) + 1, dtype=float)

As shown, the assembled polynomial with first-order fixed sparsity exhibits slower convergence than standard GMRES(6). This behavior is expected due to two primary sources of approximation error:
1. Restricting the sparsity pattern to that of $A$ is highly limiting, as the exact polynomial would naturally possess the sparsity pattern of $A^6$.
2. GMRES(6) minimizes the residual dynamically at each iteration, whereas the stationary polynomial is constructed once using a random right-hand side.

However, evaluating the computational work required for convergence provides a different perspective. When measured in terms of equivalent matrix-vector products with $A$, the assembled approach demonstrates its efficiency.

In [ ]:
# Work-based comparison for Point 3: assembled (6,1) vs GMRES(6)-apply
nnz_P_default = assembled_pattern_nnz_from_sparsity_order(A, 1)
mv_per_it_assembled = richardson_matvec_equiv_per_iter(
    nnz_A, nnz_P=nnz_P_default, matrix_free=False
)
mv_per_it_gmres6_apply = 1.0 + 6.0

work_pflare = mv_per_it_assembled * np.arange(1, len(r_rich_pflare) + 1, dtype=float)
work_gmres6_apply = mv_per_it_gmres6_apply * np.arange(1, len(r_rich_gmres6) + 1, dtype=float)

print(f'assembled (order=6, sparsity=1): matvec-equiv/iter={mv_per_it_assembled:.3f}')
print(f'GMRES(6)-apply + Richardson: matvec-equiv/iter={mv_per_it_gmres6_apply:.1f}')

plot_histories_vs_work(
    [
        ('Richardson + PCPFLAREINV (assembled 6,1)', work_pflare, r_rich_pflare),
        ('Richardson + GMRES(6) apply', work_gmres6_apply, r_rich_gmres6),
    ],
    title='Assembled (6,1) vs GMRES(6)- by matvec-equivalent work'
 )

When evaluating the computational work, the assembled fixed-sparsity approximation outperforms the application of GMRES(6). This efficiency is the primary motivation for introducing the fixed-sparsity approximation.

## 4. Increasing sparsity order

With the polynomial order fixed at 6 (assembled), increasing the `-pc_pflareinv_sparsity_order` parameter expands the allowable non-zero pattern in the assembled approximate inverse.

Mechanistically:
- `-pc_pflareinv_sparsity_order=1` maintains a strictly local sparsity pattern.
- Higher sparsity orders permit longer-range couplings, corresponding to entries with greater graph distances in the powers of $A$.
- This generally yields a more accurate approximation of the full polynomial action, thereby improving Richardson convergence.

Instead of focusing solely on iteration counts, we plot the equivalent number of matrix-vector products with $A$. Note that as the sparsity order increases, the number of non-zeros in the assembled approximation also grows.

In [ ]:
sparsity_orders = [1, 2, 3, 4, 5, 6]
curves_sparse = []
nnz_sparse = []
nnz_cache = {}

for s in sparsity_orders:
    opts_s = {
        'pc_pflareinv_sparsity_order': s,
    }

    r_s, it_s, _ = run_solver_with_history(
        A, b, ksp_type='richardson', pc_type='pflareinv',
        pflare_opts=opts_s, max_it=200, rtol=1e-10
    )

    nnz_P_s = assembled_pattern_nnz_from_sparsity_order(A, s, cache=nnz_cache)
    mv_per_it_s = richardson_matvec_equiv_per_iter(nnz_A, nnz_P=nnz_P_s, matrix_free=False)
    work_s = mv_per_it_s * np.arange(1, len(r_s) + 1, dtype=float)

    curves_sparse.append((f'sparsity_order={s}', work_s, r_s))
    nnz_sparse.append(nnz_P_s)
    print(f'sparsity_order={s}: iters={it_s}, nnz(P)={nnz_P_s}, matvec-equiv/iter={mv_per_it_s:.3f}')

plot_histories_vs_work(
    curves_sparse,
    title='Assembled PCPFLAREINV, order 6: effect of sparsity_order'
 )

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sparsity_orders, nnz_sparse, '-o')
ax.set_xlabel('sparsity_order')
ax.set_ylabel('nnz of assembled polynomial inverse P')
ax.set_title('Assembled inverse fill increases with sparsity_order')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

First, the iteration count decreases as `-pc_pflareinv_sparsity_order` increases, indicating that the approximation more closely resembles the true polynomial. Second, the number of non-zeros in the assembled approximation grows significantly with higher sparsity orders.

Consequently, employing a higher sparsity order does not necessarily yield a computationally cheaper method for this problem. The results demonstrate that more overall work is required to reach a given tolerance when using higher sparsity orders.

## 5. Increase polynomial order with fixed sparsity=1

While it may seem counterintuitive to use a polynomial order that exceeds the fixed sparsity order, this is a key feature of the assembled approximation. In this test, we fix `-pc_pflareinv_sparsity_order=1` and increase the polynomial order from 1 to 6.

This isolates the effect of the polynomial degree while maintaining a constant storage footprint. Essentially, we enrich the polynomial model without introducing additional fill-in to the assembled matrix.

Interpretation:
- Higher polynomial orders provide greater approximation power to capture inverse-like behavior.
- The fixed sparsity pattern prevents excessive memory consumption.
- As a result, we observe order-driven improvements in convergence even under strict storage constraints.

In [ ]:
orders = [1, 2, 3, 4, 5, 6, 7, 8]
curves_order_fixed_sparse = []

for m in orders:
    opts_m = {
        'pc_pflareinv_poly_order': m,
    }

    r_m, it_m, _ = run_solver_with_history(
        A, b, ksp_type='richardson', pc_type='pflareinv',
        pflare_opts=opts_m, max_it=200, rtol=1e-10
    )

    nnz_P_m = assembled_pattern_nnz_from_sparsity_order(A, 1)
    mv_per_it_m = richardson_matvec_equiv_per_iter(nnz_A, nnz_P=nnz_P_m, matrix_free=False)
    work_m = mv_per_it_m * np.arange(1, len(r_m) + 1, dtype=float)

    curves_order_fixed_sparse.append((f'order={m}, sparsity=1', work_m, r_m))
    print(f'order={m}: iters={it_m}, nnz(P)={nnz_P_m}, matvec-equiv/iter={mv_per_it_m:.3f}')

plot_histories_vs_work(
    curves_order_fixed_sparse,
    title='Assembled PCPFLAREINV: fixed sparsity=1, increasing order'
 )

The iteration count begins to plateau, suggesting that a 6th-order polynomial with 1st-order sparsity represents an effective balance. Because they share the same sparsity pattern, all assembled approximations in this test have identical application costs. However, it is important to note that the initial assembly of higher-order polynomials is computationally more expensive.

## 6. Matrix-free polynomial application

A significant advantage of polynomial approximate inverses is their ability to be applied matrix-free, eliminating the need to store an assembled approximation. We default to an assembled approximation because an explicit matrix is required to form approximate ideal restrictors in the AIRG method, which will be covered in a subsequent notebook.

Here, we enable the matrix-free application to directly compare it against standard GMRES and the previously discussed GMRES(6) applied within an undamped Richardson iteration. This isolates the performance differences between stationary and non-stationary polynomial approximations.

In [ ]:
mf_orders = [1, 2, 4, 6]

# Recompute GMRES baseline locally for this exact comparison
r_gmres_local, it_gmres_local, _ = run_solver_with_history(
    A, b, ksp_type='gmres', pc_type='none', max_it=300, rtol=1e-10
)
work_gmres_local = np.arange(1, len(r_gmres_local) + 1, dtype=float)
curves_mf = [('GMRES', work_gmres_local, r_gmres_local)]
print(f'GMRES: iters={it_gmres_local}, matvec-equiv/iter=1.0')

r_asm6, it_asm6, _ = run_solver_with_history(
    A, b, ksp_type='richardson', pc_type='pflareinv', max_it=200, rtol=1e-10
)
nnz_P_asm6 = assembled_pattern_nnz_from_sparsity_order(A, 1)
mv_per_it_asm6 = richardson_matvec_equiv_per_iter(nnz_A, nnz_P=nnz_P_asm6, matrix_free=False)
work_asm6 = mv_per_it_asm6 * np.arange(1, len(r_asm6) + 1, dtype=float)
curves_mf.append(('assembled order=6, sparsity=1', work_asm6, r_asm6))
print(f'assembled order=6, sparsity=1: iters={it_asm6}, matvec-equiv/iter={mv_per_it_asm6:.3f}')

# Add GMRES(6) apply with outer Richardson on same work axis
r_gmres6_apply, it_gmres6_apply, _ = run_richardson_with_fixed_gmres_apply(
    A, b, gmres_steps=6, max_it=200, rtol=1e-10
)
mv_per_it_gmres6_apply = 1.0 + 6.0
work_gmres6_apply = mv_per_it_gmres6_apply * np.arange(1, len(r_gmres6_apply) + 1, dtype=float)
curves_mf.append(('GMRES(6) apply + outer Richardson', work_gmres6_apply, r_gmres6_apply))
print(f'GMRES(6) apply + Richardson: iters={it_gmres6_apply}, matvec-equiv/iter={mv_per_it_gmres6_apply:.1f}')

for m in mf_orders:
    opts_mf = {
        'pc_pflareinv_poly_order': m,
        'pc_pflareinv_matrix_free': 1,
    }

    r_mf, it_mf, _ = run_solver_with_history(
        A, b, ksp_type='richardson', pc_type='pflareinv',
        pflare_opts=opts_mf, max_it=200, rtol=1e-10
    )

    mv_per_it_mf = richardson_matvec_equiv_per_iter(nnz_A, matrix_free=True, poly_order=m)
    work_mf = mv_per_it_mf * np.arange(1, len(r_mf) + 1, dtype=float)
    curves_mf.append((f'matrix-free order={m}', work_mf, r_mf))
    print(f'matrix-free order={m}: iters={it_mf}, matvec-equiv/iter={mv_per_it_mf:.1f}')

plot_histories_vs_work(
    curves_mf,
    title='Matrix-free PCPFLAREINV + assembled (6,1) + GMRES(6)-apply vs GMRES (2D upwind)'
)

Several effects are visible here. First, increasing the order of the matrix-free polynomial decreases the iteration count. However, the work calculation for this specific example indicates that repeatedly applying a 1st-order matrix-free polynomial is the most efficient among the matrix-free options.

Comparing GMRES(6) directly with the matrix-free application of the 6th-order stationary polynomial reveals that both utilize 6th-order GMRES polynomials within an undamped Richardson iteration. The key distinction is that GMRES(6) dynamically minimizes the residual at each iteration, while the stationary polynomial does not.

Consequently, GMRES(6) achieves a lower iteration count and requires less overall work to converge. However, the stationary polynomial's ability to be applied using only matrix-vector products—without requiring dot products—is a critical advantage for parallel scalability, which we will explore in future notebooks.

## Summary

- Stationary GMRES polynomials serve as effective approximate inverses.
- When memory is sufficient, fixed-sparsity assembled approximations can achieve a given tolerance with minimal computational work during the solve phase.
- In memory-constrained environments, stationary polynomials can be applied matrix-free, though this typically increases the total computational work.
- Crucially, applying stationary GMRES polynomials does not require parallel dot products.

## 7. What's next?

We will now transition to constructing a multigrid hierarchy.

**[03_cf_splitting.ipynb](03_cf_splitting.ipynb)**: Visualize the C/F splitting and explore the PMISR-DDC algorithm.